In [ ]:
!pip install -q transformers datasets tf-keras

In [ ]:
import os

# --- CRITICAL FIX FOR TENSORFLOW 2.16+ ---
# This forces TensorFlow to use the legacy Keras 2 backend, which
# is compatible with Hugging Face Transformers.
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.metrics import SparseCategoricalAccuracy
from google.colab import files


In [ ]:
BATCH_SIZE = 16
MAX_LEN = 256
LEARNING_RATE = 2e-5
EPOCHS = 2

# Verify that we are running with the fix
print(f"TensorFlow Version: {tf.__version__}")
try:
    # Check if legacy keras is active
    import tf_keras
    print("Success: Using Legacy Keras (tf-keras) compatibility.")
except ImportError:
    print("Warning: tf-keras not found. Code might fail.")

In [ ]:
print("Loading IMDB dataset...")
dataset = load_dataset("imdb")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [ ]:
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN
    )

print("Tokenizing data...")
tokenized_datasets = dataset.map(tokenize_batch, batched=True)

In [ ]:
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format(type="tensorflow", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
train_tf = tokenized_datasets["train"].to_tf_dataset(
    columns=["input_ids", "attention_mask"],
    label_cols=["labels"],
    shuffle=True,
    batch_size=BATCH_SIZE,
)

test_tf = tokenized_datasets["test"].to_tf_dataset(
    columns=["input_ids", "attention_mask"],
    label_cols=["labels"],
    shuffle=False,
    batch_size=BATCH_SIZE,
)

In [ ]:
print("Initializing Model...")
model = TFAutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss=SparseCategoricalCrossentropy(from_logits=True),
    metrics=[SparseCategoricalAccuracy(name="accuracy")]
)
model.summary()

In [ ]:
print("Starting Training...")
history = model.fit(
    train_tf,
    validation_data=test_tf,
    epochs=EPOCHS
)

In [ ]:
print("Evaluating on Test Set...")
eval_res = model.evaluate(test_tf)
print(f"Final Test Loss: {eval_res[0]}")
print(f"Final Test Accuracy: {eval_res[1]}")